# 6장 모델 자체의 추론과 도구 호출을 이용한 ReAct

TinyAgent가 추론과 도구 호출을 반복해 작업을 수행합니다.

[Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/)와 [Jay Alammar](https://www.linkedin.com/in/jalammar)의 도서 [An Illustrated Guide to AI Agents](https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/)에 수록된 6장 실습입니다. [전체 실습 목차](../README.md) · [원본 저장소](https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents)

### Google Colab에서 패키지 설치하기 (선택)

Colab 등 클라우드 노트북에서는 아래 설치 셀의 주석을 해제하고 실행합니다. 원격 모델 API를 사용한다면 Ollama 설치 셀은 건너뜁니다.

In [1]:
# %%capture
# !pip install illustrated-agents

Colab 안에서 Ollama로 모델을 실행하려면 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 → T4 GPU**를 선택한 뒤, 아래 셀의 주석을 해제하고 실행합니다.

In [2]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

---

## 1. 언어 모델 선택

내장 도구 호출과 추론을 지원하는 Gemma 4를 사용해 여러 단계의 작업을 실행합니다.

In [3]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 4 E4B(내장 추론과 도구 호출 지원)
llm = LLM(model="gemma4:e4b", think=True)

다른 언어 모델을 사용하려면 아래에서 로컬 서버 또는 클라우드 API에 해당하는 설정을 선택합니다.

In [4]:
# # llama.cpp
# llm = LLM(model="gemma-4-e4B-it-Q4_K_M", base_url="http://127.0.0.1:8080")

# # LM Studio
# llm = LLM(model="gemma-4-e4b-it", base_url="http://127.0.0.1:1234/v1")

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-4-31b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

## 2. 내장 추론과 도구 호출을 이용한 반복 실행

[앞 실습](chapter06.ipynb)에서는 모델에게 `THOUGHT`와 `ACTION`을 출력하도록 지시하고 도구 결과를 `OBSERVATION`으로 전달했습니다. 이 방법은 모델이 JSON이나 XML 형식을 잘못 출력하면 정규식으로 해석하기 어렵습니다. 이번에는 3장의 내장 추론과 5장의 내장 도구 호출을 조합합니다.

| 이전 방식 | 내장 기능을 사용하는 방식 |
|---|---|
| `THOUGHT`에서 추론 읽기 | `Response.reasoning`과 `NativeReAct` 사용 |
| `ACTION`에서 도구 호출 읽기 | `Response.tool_call`과 `NativeTools` 사용 |
| `OBSERVATION`으로 결과 전달 | `memory.add("tool", observation)`으로 도구 메시지 추가 |

먼저 `NativeReAct`를 구현합니다.

## 3. `NativeReAct` 구현

모델과 서버가 추론 및 도구 호출을 구분하므로 별도의 출력 지침이나 파싱이 필요하지 않습니다. `NativeReAct`는 한 작업의 최대 단계 수를 정하고, `Response`를 그대로 반환합니다. 추가 프롬프트는 비워 둡니다.

In [5]:
from illustrated_agents.chapters.ch6 import ReAct

class NativeReAct(ReAct):
    """모델 자체의 추론 기능을 사용하는 ReAct이다."""

    @property
    def prompt(self) -> str:
        return ""

    def parse(self, response):
        return response

이제 `NativeReAct`와 `NativeTools`를 조합해 모델이 제공하는 추론과 도구 호출을 사용할 수 있습니다.

## 4. `TinyAgent` 실행

`Tools` 대신 `NativeTools`를, `ReAct` 대신 `NativeReAct`를 전달해 에이전트를 초기화합니다. 에이전트를 호출하는 방법은 앞 실습과 같습니다.

In [6]:
from illustrated_agents.chapters.ch4 import Memory
from illustrated_agents.chapters.ch5 import NativeTools
from illustrated_agents.chapters.ch6 import TinyAgent

from illustrated_agents.toolbox import add, multiply, subtract

# 도구 등록
tools = NativeTools()
tools.add_tool("add", add, "add(a: str, b: str)")
tools.add_tool("subtract", subtract, "subtract(a: str, b: str)")
tools.add_tool("multiply", multiply, "multiply(a: str, b: str)")

# 메모리
memory = Memory()

# ReAct
react = NativeReAct(max_steps=10)

# 에이전트 생성
agent = TinyAgent(llm=llm, tools=tools, memory=memory, planner=react)

[앞 실습](chapter06.ipynb)과 같은 작업을 실행합니다.

작업은 (4.6 + 6.685)에 4를 곱하고 그 결과에서 3.14를 빼는 계산입니다.

In [7]:
# 여러 단계의 추론이 필요한 작업
agent.run("What is (4.6 + 6.685) x 4, and then subtract 3.14 from the result?")

'The final result is 42.0.'

저자의 Gemma 4 실행에서는 더 자세한 응답이 나왔습니다. 모델 응답에 추론과 도구 호출이 어떻게 담기는지 이전 방식의 메시지와 비교합니다.

In [8]:
from rich import print

print(agent.trajectory.runs)

[
    {
        'query': 'What is (4.6 + 6.685) x 4, and then subtract 3.14 from the result?',
        'steps': [
            Step(
                thought='The user wants to perform a sequence of arithmetic operations:\n1. Calculate the sum of 
4.6 and 6.685: `4.6 + 6.685`.\n2. Multiply the result from step 1 by 4: `(4.6 + 6.685) * 4`.\n3. Subtract 3.14 from
the result of step 2: `((4.6 + 6.685) * 4) - 3.14`.\n\nI have the following tools: `add`, `subtract`, 
`multiply`.\n\nStep 1: `add(a="4.6", b="6.685")`\nStep 2: `multiply(a=<result of Step 1>, b="4")`\nStep 3: 
`subtract(a=<result of Step 2>, b="3.14")`\n\nI should chain these calls.\n\nFirst, call `add`.',
                action={'tool': 'add', 'kwargs': {'a': '4.6', 'b': '6.685'}},
                observation='11.285',
                answer=None,
                metadata=None
            ),
            Step(
                thought='The user wants to perform a calculation: `(4.6 + 6.685) x 4 - 3.14`.\n\nStep 1: Calculate 
`4.6 + 6.685`. (Already done in the previous step, result is `11.285`).\nStep 2: Calculate `11.285 * 4`.\nStep 3: 
Calculate `(result from Step 2) - 3.14`.\n\nI need to use the `multiply` tool for Step 2, and then the `subtract` 
tool for Step 3.\n\nStep 2 call: `multiply(a: "11.285", b: "4")`',
                action={'tool': 'multiply', 'kwargs': {'a': '11.285', 'b': '4'}},
                observation='45.14',
                answer=None,
                metadata=None
            ),
            Step(
                thought='The user wants to calculate two operations:\n1. `(4.6 + 6.685)`\n2. Then multiply the 
result by 4.\n3. Finally, subtract 3.14 from the result of step 2.\n\nStep 1: Add 4.6 and 6.685.\nThe previous tool
call used `add{a: "4.6", b: "6.685"}` and the response was `11.285`. This is correct.\n\nStep 2: Multiply 11.285 by
4.\nThe previous tool call used `multiply{a: "11.285", b: "4"}` and the response was `45.14`. This is 
correct.\n\nStep 3: Subtract 3.14 from 45.14.\nI need to use the `subtract` tool now.\na = 45.14\nb = 3.14',
                action={'tool': 'subtract', 'kwargs': {'a': '45.14', 'b': '3.14'}},
                observation='42.0',
                answer=None,
                metadata=None
            ),
            Step(
                thought='The user wants to calculate a multi-step arithmetic expression: `((4.6 + 6.685) * 4) - 
3.14`.\n\nI have already executed the steps using the available tools:\n\n1.  **Calculate `4.6 + 6.685`**:\n    *  
Tool call: `add(a: "4.6", b: "6.685")`\n    *   Tool response: `11.285` (This is correct: 4.6 + 6.685 = 
11.285)\n\n2.  **Calculate `11.285 * 4`**:\n    *   Tool call: `multiply(a: "11.285", b: "4")`\n    *   Tool 
response: `45.14` (This is correct: 11.285 * 4 = 45.14)\n\n3.  **Calculate `45.14 - 3.14`**:\n    *   Tool call: 
`subtract(a: "45.14", b: "3.14")`\n    *   Tool response: `42.0` (This is correct: 45.14 - 3.14 = 42.0)\n\nThe 
final result is 42.0. I should now provide this final answer to the user.',
                action=None,
                observation=None,
                answer='The final result is 42.0.',
                metadata={'model': 'gemma4:e4b', 'prompt_tokens': 328, 'completion_tokens': 334}
            )
        ]
    }
]

In [9]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

메모리를 열어 모델이 도구를 어떤 형식으로 호출했는지 확인합니다.

In [10]:
print(agent.memory.messages)

[
    {'role': 'system', 'content': 'You are a helpful assistant.\n\n'},
    {'role': 'user', 'content': 'What is (4.6 + 6.685) x 4, and then subtract 3.14 from the result?'},
    {
        'role': 'assistant',
        'content': '',
        'tool_calls': [
            {
                'id': 'call_rjflwe0i',
                'index': 0,
                'type': 'function',
                'function': {'name': 'add', 'arguments': '{"a":"4.6","b":"6.685"}'}
            }
        ]
    },
    {'role': 'tool', 'content': '11.285'},
    {
        'role': 'assistant',
        'content': '',
        'tool_calls': [
            {
                'id': 'call_qznsyyyz',
                'index': 0,
                'type': 'function',
                'function': {'name': 'multiply', 'arguments': '{"a":"11.285","b":"4"}'}
            }
        ]
    },
    {'role': 'tool', 'content': '45.14'},
    {
        'role': 'assistant',
        'content': '',
        'tool_calls': [
            {
                'id': 'call_xpqxid4g',
                'index': 0,
                'type': 'function',
                'function': {'name': 'subtract', 'arguments': '{"a":"45.14","b":"3.14"}'}
            }
        ]
    },
    {'role': 'tool', 'content': '42.0'},
    {'role': 'assistant', 'content': 'The final result is 42.0.'}
]

`assistant` 메시지의 `tool_calls`에는 호출할 함수와 인자가 들어갑니다. 함수가 반환한 값은 `tool` 메시지로 추가해 모델에 전달합니다. 이 메시지가 앞에서 직접 지정했던 `OBSERVATION`에 해당합니다.

```json
[
    ...
    {
        'role': 'assistant',
        'content': '',
        'tool_calls': [
            {
                'id': 'call_8sezthoj',
                'function': {'arguments': '{"a":"45.14","b":"3.14"}', 'name': 'subtract'},
                'type': 'function',
                'index': 0
            }
        ]
    },
    {'role': 'tool', 'content': '42.0'},
    ...
]
```

---

# 구현한 내용

프롬프트 형식을 직접 해석하던 ReAct 반복문을 모델 자체의 추론 및 도구 호출에 연결했습니다. 모델을 호출하고 결과를 다음 단계에 전달하는 루프는 재사용합니다.

In [11]:
from illustrated_agents.chapters.ch6 import what_we_built_native; what_we_built_native

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ ├── agent.py                                                                                                    │
│ ├── llm.py                                                                                                      │
│ ├── memory.py                                                                                                   │
│ ├── planning.py   ← New (Added the `NativeReAct` class)                                                         │
│ ├── toolbox.py                                                                                                  │
│ ├── tools.py                                                                                                    │
│ └── trajectory.py                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# 다음 장

[7장](../chapter07/chapter07.ipynb)에서는 언어 모델과 에이전트가 과제를 얼마나 잘 수행하는지 평가합니다.